# Stability Test

This notebook aims to check the reproducibility of the results of `diamx`.

In [ ]:
import os
import diamx
from diamx.model import DiamxModel
from diamx.utils import generate_bin_array
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch
import json

diamx_dir = os.path.dirname(diamx.__file__)

import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="appletree.config")
warnings.filterwarnings("ignore", category=UserWarning, module="appletree.component")

In [ ]:
plt.style.use("seaborn-v0_8-colorblind")
plt.rcParams.update(
    {
        "figure.figsize": (6.4, 4.8),
        "figure.dpi": 600,
        "font.family": "serif",
        "font.size": 12,
        # 'figure.dpi': 300,
        "lines.linewidth": 2.0,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "xtick.major.size": 8,
        "xtick.minor.size": 4,
        "ytick.major.size": 8,
        "ytick.minor.size": 4,
        "xtick.major.width": 1,
        "xtick.minor.width": 0.7,
        "ytick.major.width": 1,
        "ytick.minor.width": 0.7,
        "xtick.direction": "in",
        "ytick.direction": "in",
        # 'legend.loc': 'upper center',
        # 'legend.bbox_to_anchor': (0.5, 1.05),
        # 'legend.ncol': 3,
        "legend.fancybox": True,  # if True, use a rounded box for the
        # legend background, else a rectangle
        "legend.fontsize": 12,
        "text.usetex": True,
    }
)

In [ ]:
with open("../config/xenonnt_sr0_wimp_config.json", "r") as f:
    config = json.load(f)
test_config = config.copy()
test_config

In [ ]:
test_config["signal"]["parameter_range"] = [100, 10000]
from tqdm import tqdm
from diamx.utils import HiddenPrints, HiddenTqdm
from copy import deepcopy

repeat_times = 10


def update_config_for_run(config, run_index, batch_size=None):
    new_config = deepcopy(config)
    for experiment_idx, experiment_config in enumerate(config["experiments"]):
        for idx, bkg_config in enumerate(experiment_config["bkgs"]):
            if batch_size is None:
                new_config["experiments"][experiment_idx]["bkgs"][idx]["args"] = dict(
                    bkg_config["args"], **{"run_index": run_index}
                )
            else:
                new_config["experiments"][experiment_idx]["bkgs"][idx]["args"] = dict(
                    bkg_config["args"],
                    **{"run_index": run_index, "batch_size": batch_size},
                )
        for idx, shaped_bkg_config in enumerate(experiment_config["shaped_bkgs"]):
            if batch_size is None:
                new_config["experiments"][experiment_idx]["shaped_bkgs"][idx][
                    "args"
                ] = dict(shaped_bkg_config["args"], **{"run_index": run_index})
            else:
                new_config["experiments"][experiment_idx]["shaped_bkgs"][idx][
                    "args"
                ] = dict(
                    shaped_bkg_config["args"],
                    **{"run_index": run_index, "batch_size": batch_size},
                )
    if batch_size is None:
        new_config["signal"]["args"] = dict(
            config["signal"]["args"], **{"run_index": run_index}
        )
    else:
        new_config["signal"]["args"] = dict(
            config["signal"]["args"],
            **{"run_index": run_index, "batch_size": batch_size},
        )
    return new_config

In [ ]:
for run_index in tqdm(range(repeat_times)):
    test_config_this_run = update_config_for_run(
        test_config, run_index, batch_size=None
    )
    with HiddenPrints():
        with HiddenTqdm():
            st = diamx.Context(config=test_config_this_run)
            st.register_experiment(diamx.experiments.XENONnTSR0)
            st.generate_templates()
            diamx.run_inference_pool(
                st,
                processes=2,
                output_file_name=f"xenonnt_sr0_wimp_nominal_run_{run_index}.csv",
            )

In [6]:
# Retrieve results
all_upper_limits = []
for run_index in range(repeat_times):
    result = np.loadtxt(
        f"./diamx_output/xenonnt_sr0_wimp_nominal_run_{run_index}.csv", delimiter=","
    )
    all_upper_limits.append(result[:, 2])
all_upper_limits = np.array(all_upper_limits)
print("Median upper limits:", np.median(all_upper_limits, axis=0))
print("Standard deviation of upper limits:", np.std(all_upper_limits, axis=0))
print(
    f"The fluctuation is at {np.std(all_upper_limits, axis=0) / np.median(all_upper_limits, axis=0)} level."
)

Median upper limits: [0.00734892 0.6887027 ]
Standard deviation of upper limits: [4.54438737e-05 3.66814122e-03]
The fluctuation is at [0.00618375 0.00532616] level.


In [ ]:
plt.plot(all_upper_limits[:, 0] * 1e-44)
plt.axhline(
    np.median(all_upper_limits, axis=0)[0] * 1e-44,
    color="red",
    linestyle="--",
    label="Median",
)
plt.fill_between(
    np.arange(repeat_times),
    1e-44
    * (np.median(all_upper_limits, axis=0)[0] - np.std(all_upper_limits, axis=0)[0]),
    1e-44
    * (np.median(all_upper_limits, axis=0)[0] + np.std(all_upper_limits, axis=0)[0]),
    color="red",
    alpha=0.3,
    label="1 sigma band",
)
plt.legend()
plt.xlabel("Run index")
plt.ylabel("Upper limit at 100 GeV [cm$^2$]")
plt.savefig("../plots/stability_xenonnt_sr0_wimp.png", dpi=600)